In [ ]:
# !pip install sentence-transformers ftfy transformers pillow faiss-cpu numpy pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 18.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv(r"filtered_data_with_images.csv")   # your filtered data
df

FileNotFoundError: [Errno 2] No such file or directory: 'filtered_data_with_images.csv'

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

# df = pd.read_csv("sampled_150_products.csv")   # your filtered data
df = pd.read_csv("filtered_data_with_images.csv")

# Choose model
text_model = SentenceTransformer("all-MiniLM-L6-v2")  # 384d

# Make canonical text: prefer description but include title & style tags
def make_text(row):
    parts = [str(row.get("Product_Name","")), str(row.get("Product_Description",""))]
    for k in ("Style_Descriptor","Occasion_Tag","Adword_Grouping"):
        if row.get(k): parts.append(str(row[k]))
    return " ".join([p for p in parts if p])

texts = df.apply(make_text, axis=1).tolist()
text_embs = text_model.encode(texts, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
print("text_embs shape:", text_embs.shape)  # (N, 384)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2439 [00:00<?, ?it/s]

text_embs shape: (78017, 384)


In [ ]:
df.size

1482323

### Image embeddings (CLIP)

In [ ]:
from sentence_transformers import SentenceTransformer
from PIL import Image
import requests, io

clip = SentenceTransformer("clip-ViT-B-32")  # 512d

def image_to_emb(url):
    try:
        if url.startswith("http"):
            resp = requests.get(url, timeout=10)
            img = Image.open(io.BytesIO(resp.content)).convert('RGB')
        else:
            img = Image.open(url).convert('RGB')
        emb = clip.encode(img, convert_to_numpy=True, normalize_embeddings=True)
        return emb
    except Exception as e:
        return np.zeros((512,), dtype=np.float32)  # fallback

image_embs = np.vstack([image_to_emb(u) for u in df['Image_URL'].fillna("").tolist()])
print("image_embs shape:", image_embs.shape)  # (N, 512)


image_embs shape: (78017, 512)


### Categorical embeddings (example with PyTorch Embedding at training time)

In [ ]:
# Example: Subcategory embedding (offline)
import numpy as np
subcats = df['Subcategory'].fillna("Unknown").astype(str)
uniq = subcats.unique().tolist()
idx = {v:i for i,v in enumerate(uniq)}
sub_idx = np.array([idx[x] for x in subcats])

# Create random/learned embedding matrix or use a small trained model. For offline, you can use random init + normalize:
embed_dim = 32
np.random.seed(42)
subcat_emb_table = np.random.randn(len(uniq), embed_dim).astype(np.float32)
subcat_embs = subcat_emb_table[sub_idx]  # (N, 32)


### Numeric features: scaling

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num_df = df[['Price_INR', 'Discount_Percentage', 'Style_Reward_Points']].fillna(0).astype(float)
scaler = StandardScaler()
num_scaled = scaler.fit_transform(num_df)  # shape (N, 3)


### Concatenate & reduce

In [ ]:
# concat: [text_emb (384) | image_emb (512) | subcat (32) | num (3)] -> 931 dims
concat = np.hstack([text_embs, image_embs, subcat_embs, num_scaled.astype(np.float32)])
print("concat shape", concat.shape)

# optional: PCA to 256 dims
from sklearn.decomposition import PCA
pca = PCA(n_components=140, random_state=42) # Changed n_components to 140
final_vecs = pca.fit_transform(concat.astype(np.float32))  # (N,256)

# normalize before indexing
from sklearn.preprocessing import normalize
final_vecs = normalize(final_vecs, axis=1).astype(np.float32)

NameError: name 'num_scaled' is not defined

### Save vectors + metadata for FAISS

In [ ]:
np.save("product_vectors.npy", final_vecs)
df.to_parquet("products_with_meta.parquet", index=False)


In [ ]:
import faiss

# Build a FAISS index
index = faiss.IndexFlatL2(final_vecs.shape[1])  # Use L2 distance for similarity search
index.add(final_vecs)  # Add the vectors to the index

# Save the FAISS index
faiss.write_index(index, "product_vectors.faiss")

# Load the product metadata
products_df = pd.read_parquet("products_with_meta.parquet")

In [ ]:
from IPython.display import display, Image, HTML
import os
import faiss
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer # Assuming text_model is needed
import base64 # Import base64 module

# Load the necessary models and data outside the function to avoid reloading on every call
# Assuming text_model, pca, image_embs, subcat_embs, num_scaled are defined in previous cells
# If not, they should be loaded or defined here as well.

def search_products(query, k=5):
    """
    Searches for similar products based on a text query and displays their images and details in a styled grid format, including similarity score.

    Args:
        query (str): The search query.
        k (int): The number of similar products to retrieve.

    Returns:
        pandas.DataFrame: A DataFrame containing the top k similar products.
    """
    # Load the FAISS index inside the function to ensure it's available
    try:
        index = faiss.read_index("product_vectors.faiss")
    except Exception as e:
        print(f"Error loading FAISS index: {e}")
        return None

    # Load the product metadata inside the function to ensure it's available
    try:
        products_df = pd.read_parquet("products_with_meta.parquet")
    except Exception as e:
        print(f"Error loading product metadata: {e}")
        return None

    # Embed the query
    # Ensure text_model is accessible (defined globally or passed as argument)
    if 'text_model' not in globals():
        print("Error: text_model is not defined. Please run the cell that initializes SentenceTransformer.")
        return None
    query_emb = text_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)

    # Perform PCA on the query embedding (using the same PCA object fitted on the product vectors)
    # Ensure pca, image_embs, subcat_embs, num_scaled are accessible
    if 'pca' not in globals() or 'image_embs' not in globals() or 'subcat_embs' not in globals() or 'num_scaled' not in globals():
         print("Error: Required variables (pca, image_embs, subcat_embs, num_scaled) are not defined. Please run the relevant cells.")
         return None

    # Create a dummy array for concatenation with the query embedding
    # The shape of the dummy array should match the dimensions of image_embs, subcat_embs, and num_scaled
    # assuming the query embedding only replaces the text_emb part.
    dummy_concat_part = np.zeros((1, image_embs.shape[1] + subcat_embs.shape[1] + num_scaled.shape[1]), dtype=np.float32)
    query_concat = np.hstack([query_emb, dummy_concat_part]).astype(np.float32)


    query_pca = pca.transform(query_concat)

    # Normalize the query embedding
    query_pca = normalize(query_pca, axis=1).astype(np.float32)

    # Search the FAISS index
    distances, indices = index.search(query_pca, k)

    # Retrieve the similar products from the dataframe
    similar_products = products_df.iloc[indices[0]].copy() # Use .copy() to avoid SettingWithCopyWarning
    similar_products['Distance'] = distances[0] # Add distances to the dataframe


    # Display the images and details in a styled grid format
    image_folder = "/content/uploaded_images"

    # Start the HTML for the grid container
    grid_html = """
    <div style="display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 20px;">
    """

    for i, (index, row) in enumerate(similar_products.iterrows()):
        sku = str(row['SKU_No'])
        distance = row['Distance'] # Get the distance for this product
        # Try with original SKU first
        image_filename = f"{sku}.jpg"
        image_path = os.path.join(image_folder, image_filename)

        # If not found, try with a leading zero
        if not os.path.exists(image_path):
            image_filename = f"0{sku}.jpg"
            image_path = os.path.join(image_folder, image_filename)

        # Build the HTML for each product item
        grid_html += f"""
        <div style="border: 1px solid #e0e0e0; border-radius: 5px; padding: 15px; text-align: center;">
        """

        if os.path.exists(image_path):
            grid_html += f"""
                <img src="data:image/jpeg;base64,{base64.b64encode(open(image_path, 'rb').read()).decode()}" style="width: 100%; height: auto; border-radius: 4px; margin-bottom: 10px;">
            """
        else:
            grid_html += f"""
                <div style="width: 100%; height: 150px; background-color: #f0f0f0; display: flex; justify-content: center; align-items: center; border-radius: 4px; margin-bottom: 10px;">
                    Image not found
                </div>
            """

        grid_html += f"""
            <h4 style="margin-top: 0; margin-bottom: 5px; color: #333; font-size: 1.1em;">{row['Product_Name']}</h4>
            <p style="margin-bottom: 5px; color: #555; font-size: 0.9em;"><b>SKU:</b> {sku}</p>
            <p style="margin-bottom: 5px; color: #555; font-size: 0.9em;"><b>Category:</b> {row['Category']} | <b>Subcategory:</b> {row['Subcategory']}</p>
            <p style="margin-bottom: 5px; color: #555; font-size: 0.9em;"><b>Price:</b> ₹{row['Price_INR']} | <b>Discount:</b> {row['Discount_Percentage']}%</p>
            <p style="margin-bottom: 5px; color: #555; font-size: 0.9em;"><b>Stock:</b> {row['Stock_Status']}</p>
            <p style="margin-bottom: 5px; color: #555; font-size: 0.9em;"><b>Distance:</b> {distance:.4f}</p>
            <p style="font-size: 0.8em; color: #777; overflow: hidden; text-overflow: ellipsis; display: -webkit-box; -webkit-line-clamp: 3; -webkit-box-orient: vertical; text-align: left;">{row['Product_Description']}</p>
        </div>
        """

    # Close the grid container HTML
    grid_html += "</div>"

    # Display the entire grid
    display(HTML(grid_html))

    return similar_products

# Example usage:
query = "blue t-shirt"
similar_items = search_products(query)
# No need to display the DataFrame again here as details are shown per product
# if similar_items is not None:
#     display(similar_items)

**Sample Query 1: Searching by product description**

In [ ]:
query_sample_1 = "Red WC shorts for Men Party"
print(f"Searching for: {query_sample_1}")
search_products(query_sample_1)

**Sample Query 2: Searching by category and style**

In [ ]:
query_specific_product = "Garment Lower body Shorts Red Men Placement print Cotton Blend Spring Party"
print(f"Searching for products matching attributes: {query_specific_product}")
search_products(query_specific_product)

In [ ]:
query_sample_2 = " white Sleeveless top"
print(f"Searching for: {query_sample_2}")
search_products(query_sample_2)

**Sample Query 3: Searching by specific attributes**

In [ ]:
query_sample_3 = "Angela top"
print(f"Searching for: {query_sample_3}")
search_products(query_sample_3)

In [ ]:
!unzip /content/images_filtered.zip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob

# Create a new directory to store the images
image_folder = "/content/uploaded_images"
os.makedirs(image_folder, exist_ok=True)

# Move image files to the new folder
image_extensions = ['*.jpg', '*.png', '*.jpeg', '*.gif', '*.bmp']
for ext in image_extensions:
    for file_path in glob.glob(f"/content/{ext}"):
        os.rename(file_path, os.path.join(image_folder, os.path.basename(file_path)))

print(f"Moved image files to: {image_folder}")

In [ ]:
from IPython.display import display, HTML
import base64, os

def search_products(query, k=5):
    index = faiss.read_index("product_vectors.faiss")
    products_df = pd.read_parquet("products_with_meta.parquet")

    query_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    query_pca = pca.transform(query_emb)
    query_pca = normalize(query_pca, axis=1).astype(np.float32)

    distances, indices = index.search(query_pca, k)
    similar = products_df.iloc[indices[0]].copy()
    similar["Distance"] = distances[0]

    grid_html = """<div style='display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 20px;'>"""

    for _, row in similar.iterrows():
        img_url = str(row.get("Image_URL", ""))
        grid_html += f"""
        <div style='border: 1px solid #ccc; border-radius: 6px; padding: 10px; text-align: center;'>
            <img src='{img_url}' style='width: 100%; height: auto; border-radius: 6px; margin-bottom: 8px;'>
            <h4 style='margin: 5px 0;'>{row['Product_Name']}</h4>
            <p style='margin: 3px 0; color: #666; font-size: 0.9em;'><b>Category:</b> {row['Category']} | {row['Subcategory']}</p>
            <p style='margin: 3px 0; color: #666; font-size: 0.9em;'><b>Price:</b> ₹{row['Price_INR']} | <b>Discount:</b> {row['Discount_Percentage']}%</p>
            <p style='margin: 3px 0; color: #666; font-size: 0.9em;'><b>Distance:</b> {row['Distance']:.4f}</p>
        </div>"""

    grid_html += "</div>"
    display(HTML(grid_html))
    return similar


In [ ]:
search_products("red sleeveless tank top for men", k=5)

,SKU_No,Product_Name,Product_Description,Category,Subcategory,Primary_Colour_Family,Gender_Target,Style_Descriptor,Image_URL,Material_Composition,Season,Occasion_Tag,Eco_Score,Style_Reward_Points,Adword_Grouping,Stock_Status,Price_INR,Discount_Percentage,Return_Type,Distance
0,1,3p fancy tanktop body,Sleeveless bodysuits in soft organic cotton je...,Garment Upper body,Vest top,White,Men,All over pattern,images/034/0348995028.jpg,Cotton Blend,Spring,Sports,85,40,garment_upper_body,Out of Stock,4896,14,15 days return,1.183894
58,59,mister muscle SS,Short-sleeved shirt in a stretch cotton weave ...,Garment Upper body,Shirt,Black,Women,All over pattern,images/061/0618329008.jpg,Cotton Blend,Spring,Casual,65,35,garment_upper_body,Low Stock,4908,51,No return,1.412632
46,47,Justin 2-p tank SB,Vest tops in cotton jersey. One with an all-ov...,Garment Upper body,Vest top,Red,Women,Mixed solid/pattern,images/048/0485546020.jpg,Cotton Blend,Spring,Casual,96,55,garment_upper_body,Out of Stock,1080,8,15 days return,1.415964
97,98,Milano,"Fitted, turtleneck top in ribbed jersey with l...",Garment Upper body,Sweater,Grey,Teen Girls,Solid,images/079/0790651002.jpg,Cotton Blend,Summer,Formal,60,61,garment_upper_body,Low Stock,2875,32,No return,1.603246
28,29,Poppins,"Short, sleeveless dress in slub jersey made fr...",Garment Full body,Dress,Black,Men,Solid,images/066/0667533002.jpg,Cotton Blend,Spring,Work,57,16,garment_full_body,Low Stock,4254,64,7 days return,1.641643
